# Anomaly check

QC on satellite GVF vs PhenoCam GCC and NDVI: scores table, gap-by-veg boxplots,
and a golden-standard ranking.

**Spin-up** (`gvf_sos == 1`): the phenology fit failed and landed on DOY 1 by
accident, not because green-up really started on Jan 1. On flat, low amplitude
curves (EN, sparse shrub, evergreen) there is no clear winter to summer swing, so
it pin SOS at the first day of data. That inflates gap /
divergence vs NDVI or GCC (noise misread as signal), so spin-up sites must be
flagged and usually excluded before interpreting lag or compression.

Artifacts: `anomaly_pipeline/output/` (`metadata/` scores, `boxplot/`,
`golden_standard_ranking.csv`).

**Veg Codes** DB = deciduous broadleaf, EN = evergreen needle, GR = grassland, AG = agriculture, SH = shrub

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
if REPO.name == "anomaly_pipeline":
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared.data_collection import (
    build_golden_ranking,
    collect_folder,
    group_summary,
    load_table,
    plot_gap_boxplot_by_veg,
    top_n,
)

# GVF text in plotting stage (drop once, reuse here)
INPUT_DIR = REPO / "plotting_pipeline" / "input"
ANOMALY_DIR = REPO / "anomaly_pipeline" / "output"
METADATA_DIR = ANOMALY_DIR / "metadata"

## MetaData table

One row per site-year: SOS/MOS/DOS/EOS for GVF, GCC, and NDVI, plus pairwise
gap / DTW / divergence. Use it to find spin-up (`gvf_sos == 1`), large land-type
offsets, and other bad fits without opening every plot.

Writes `anomaly_pipeline/output/metadata/<FOLDER>_scores.csv`.

In [ ]:
FOLDER = "GBOV_2023"  # GBOV_2024 / GoldenSites_2023
LIMIT = None
SORT = "gvf_vs_ndvi_div"
TOP = 10

csv_path = collect_folder(FOLDER, INPUT_DIR, ANOMALY_DIR, limit=LIMIT)
df = load_table(csv_path)
print(csv_path, "|", len(df), "rows")
df.head()

In [ ]:
cols = [
    "site", "veg", "year",
    "gvf_sos", "gcc_sos", "ndvi_sos",
    "gvf_vs_ndvi_div", "gvf_vs_ndvi_gap", "gvf_vs_ndvi_dtw",
    "gvf_vs_gcc_div", "gcc_vs_ndvi_div",
]
cols = [c for c in cols if c in df.columns]
display(top_n(df, by=SORT, n=TOP)[cols])
display(group_summary(df, by="veg"))

## Satellite Data Gap boxplot by veg

Distribution of `gvf_vs_ndvi_gap` by vegetation type. Spin-up sites are red
diamonds so we can see how much they inflate the apparent discrepancy
(especially EN / GR; DB barely moves). 

True lag is a consistent date offset across many sites of the same veg that does **not** track the spin-up flag.
True compression is a real shrink/stretch of green-up to peak shape (DTW is
more sensitive to that than a single date gap).

Writes `anomaly_pipeline/output/boxplot/<FOLDER>_BOXPLOT.png`.

In [ ]:
boxplot_path = plot_gap_boxplot_by_veg(csv_path, ANOMALY_DIR)
print(boxplot_path)

## Golden standard ranking

Drop spin-up, then rank sites by combined GVF-GCC / GVF-NDVI divergence (gap +
DTW). Closed-canopy **DB** sites are flagged as the control group: most uniform
at VIIRS scales, tightest cross-product agreement, almost no spin-up. Their
gap/DTW distribution is the irreducible baseline under ideal conditions. Only
*excess* discrepancy in shrub / mixed forest beyond that baseline can be
attributed to land-cover-driven lag, not the raw number alone.

Needs scores under `output/metadata/`. Writes `output/golden_standard_ranking.csv`.

In [ ]:
rank_path = build_golden_ranking(ANOMALY_DIR)
rank = load_table(rank_path)
rank_cols = [
    "rank", "site", "veg", "year", "source", "golden_candidate",
    "combined_div", "combined_gap", "combined_dtw",
]
rank_cols = [c for c in rank_cols if c in rank.columns]
print(rank_path, "|", len(rank), "rows |", int(rank["golden_candidate"].sum()), "DB candidates")
display(rank.head(15)[rank_cols])
display(rank.loc[rank["golden_candidate"]].head(15)[rank_cols])